In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

def generate_synthetic_water_data():
    start_time = datetime(2025, 1, 1, 0, 0, 0)
    end_time = start_time + timedelta(days=90)  # 3 months
    interval = 5 * 60  # 5 minutes = 300 seconds

    total_points = int((end_time - start_time).total_seconds() / interval)
    print(f"Generating {total_points:,} data points (~{total_points/90000:.1f} days worth)...")

    # Generate timestamps
    timestamps = [start_time + timedelta(seconds=i * interval) for i in range(total_points)]

    # Baseline data
    temperature = np.random.normal(30, 1, total_points)
    ph = np.random.normal(7.2, 0.2, total_points)
    turbidity = np.random.normal(3, 0.5, total_points)
    tds = np.random.normal(350, 40, total_points)

    # Label columns
    dump_detected = np.zeros(total_points)
    mining_dump = np.zeros(total_points)
    paper_dump = np.zeros(total_points)
    chemical_dump = np.zeros(total_points)
    sewage_dump = np.zeros(total_points)

    # Determine number of dump events (roughly 2–3 per week)
    total_dumps = random.randint(150, 250)
    dump_indices = random.sample(range(1000, total_points-1000), total_dumps)

    for idx in dump_indices:
        dump_detected[idx:idx+60] = 1  # lasts ~4 mins
        dump_type = random.choice(["mining", "paper", "chemical", "sewage", "mixed"])

        if dump_type == "mining":
            ph[idx:idx+60] -= np.random.uniform(1.0, 2.0)
            mining_dump[idx:idx+60] = 1

        elif dump_type == "paper":
            ph[idx:idx+60] += np.random.uniform(1.0, 2.0)
            paper_dump[idx:idx+60] = 1

        elif dump_type == "chemical":
            tds[idx:idx+60] += np.random.uniform(400, 800)
            chemical_dump[idx:idx+60] = 1

        elif dump_type == "sewage":
            turbidity[idx:idx+60] += np.random.uniform(8, 15)
            sewage_dump[idx:idx+60] = 1

        elif dump_type == "mixed":
            ph[idx:idx+60] += np.random.uniform(-1.5, 1.5)
            tds[idx:idx+60] += np.random.uniform(300, 900)
            turbidity[idx:idx+60] += np.random.uniform(5, 10)
            chemical_dump[idx:idx+60] = sewage_dump[idx:idx+60] = 1

        # Add temperature noise during dump
        temperature[idx:idx+60] += np.random.uniform(-0.5, 0.5)

    # Postprocess - clamp values to realistic ranges
    ph = np.clip(ph, 4.0, 10.0)
    turbidity = np.clip(turbidity, 0, 100)
    tds = np.clip(tds, 100, 2000)
    temperature = np.clip(temperature, 20, 40)

    df = pd.DataFrame({
        "timestamp": timestamps,
        "temperature": temperature,
        "ph": ph,
        "turbidity": turbidity,
        "tds": tds,
        "dump_detected": dump_detected,
        "mining_dump_detected": mining_dump,
        "paper_dump_detected": paper_dump,
        "chemical_dump_detected": chemical_dump,
        "sewage_dump_detected": sewage_dump
    })

    # Save to CSV
    df.to_csv("synthetic_water_quality_3months.csv", index=False)
    print("✅ Synthetic dataset saved as 'synthetic_water_quality_3months.csv'")
    print(df.head())

if __name__ == "__main__":
    generate_synthetic_water_data()



Generating 25,920 data points (~0.3 days worth)...
✅ Synthetic dataset saved as 'synthetic_water_quality_3months.csv'
            timestamp  temperature        ph  turbidity         tds  \
0 2025-01-01 00:00:00    30.097569  7.119171   2.037818  341.972505   
1 2025-01-01 00:05:00    30.172181  7.112464   2.761623  350.366125   
2 2025-01-01 00:10:00    30.092684  7.386127   2.915353  274.190132   
3 2025-01-01 00:15:00    27.834384  6.829116   2.599621  300.393739   
4 2025-01-01 00:20:00    31.309639  7.318417   2.536540  391.789683   

   dump_detected  mining_dump_detected  paper_dump_detected  \
0            0.0                   0.0                  0.0   
1            0.0                   0.0                  0.0   
2            0.0                   0.0                  0.0   
3            0.0                   0.0                  0.0   
4            0.0                   0.0                  0.0   

   chemical_dump_detected  sewage_dump_detected  
0                     0.0 